# CS781 Phase 3 - Rhetorical Role (RR) - Codabench submission generator (Model 1 + LoRA / LoRMA)

Makes the **Codabench submission** from an already trained run. No training happens here.

**Attach as inputs**
1. Output of the **RR data-prep notebook run AFTER the test set was released** (`tokenized_rr.pkl` with test documents, `rr_utils.py`).
2. Output of your **Model-1 Phase-1 notebook** (`legal-gqa-final`).
3. A place that holds **`legal_gqa_model.py`** (for example the Phase-2 Model-1 NER notebook output).
4. The **output of the training notebook**: folder `rr_model1_gqa_lora/` (or `rr_model1_gqa_lorma/` for LoRMA) with `fold0 ... fold4` (`best.pt`, `oof_logp.pkl`) and `cv_state.json`.

Set `ADAPTER = "lora"` or `"lorma"` below so it matches the run you attached. Settings must be the same as in the training notebook (they are the defaults).

**What it does**
1. Loads the 5 fold models (Phase-1 backbone + the saved adapter / head weights of each fold).
2. Predicts every test document with each fold model (3 different chunk cuttings each) and averages the log-probabilities.
3. Re-tunes the prior correction `tau` and the Viterbi weight `alpha` on the saved out-of-fold predictions and decodes with them.
4. Checks every prediction (document ids, one label per sentence, labels in the vocabulary) and writes `prediction.json`, `prediction_CL.json`, `prediction_IT.json` and `submission.zip`.

Outputs: `<run name>_submission/`. Upload `submission.zip` (or the json files, see the Codabench page for the exact rule of Phase 3).

## 1. Settings

In [ ]:
import os, sys, glob, json, time, math, random, shutil, pickle, re, hashlib, zipfile, types
import numpy as np

T0 = time.time()

# ======================= SETTINGS =======================
ADAPTER = "lora"            # "lora" or "lorma": must match the training run you attached
RUN_NAME = None             # folder name of the training run; None = rr_model1_gqa_lora / rr_model1_gqa_lorma
SMOKE_TEST = False          # True = tiny fast run to check that everything works

PROTOCOL = "kfold"          # "kfold" (recommended) or "dev" (train on train, validate on dev)
N_FOLDS = 5
SEED = 42

# --- chunks ---
TRAIN_CHUNK_MIN = 512       # each epoch every document is re-cut with a random chunk length in [MIN, MAX]
TRAIN_CHUNK_MAX = 1024      # (different boundaries every epoch = augmentation). Must be <= Phase-1 context (1024)
INFER_CHUNK_LENS = [1024, 800, 640]   # test/OOF: average the log-probs of these different cuttings

# --- training ---
EPOCHS = 8
PATIENCE = 3                # stop a fold if validation macro-F1 does not improve for this many epochs
BATCH_CHUNKS = 4            # chunks per optimizer step
MICRO_BATCH = 2             # chunks per forward pass (lower it if you run out of GPU memory)
EVAL_BATCH = 4
LR_LORA = 2e-4            # (training setting, not used here)
LR_LORMA = 5e-4
LR_HEAD = 1e-3
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.06
GRAD_CLIP = 1.0
LABEL_SMOOTHING = 0.02
USE_CHECKPOINT = True       # gradient checkpointing (saves GPU memory)

# --- class imbalance ---
CLASS_WEIGHTS = "sqrt"      # "none" | "sqrt" | "inv"   (computed from the training documents of each fold)
WEIGHT_CAP = 10.0           # largest allowed class weight
OVERSAMPLE_RARE = True      # repeat chunks that contain a rare label
RARE_MAX_COUNT = 320        # a label with at most this many training sentences is "rare"
OVERSAMPLE_EXTRA = 1        # how many extra copies of such chunks

# --- LoRA ---
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# --- LoRMA (only used when ADAPTER = "lorma") ---
LORMA_VARIANT = "plus"
LORMA_R = 16
LORMA_ALPHA = 16
LORMA_DROPOUT = 0.05
LORMA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# --- head ---
HEAD = "bigru"              # "bigru" or "linear" (linear = no cross-sentence head, for the ablation)
USE_FEATURES = True         # position + legal cue features
HEAD_DIM = 512
GRU_HIDDEN = 256
HEAD_DROPOUT = 0.2
DOMAIN_EMB = 16

# --- post-processing (tuned on out-of-fold predictions) ---
TAU_GRID = [-0.5, -0.25, 0.0, 0.25, 0.5, 0.75, 1.0]   # prior correction strength
ALPHA_GRID = [0.0, 0.25, 0.5, 1.0]                    # transition weight in Viterbi

# --- session ---
MAX_SESSION_HOURS = 10.0    # stop cleanly after this long (Kaggle commit limit is 12 h); progress is saved

# --- paths (None = search /kaggle/input automatically) ---
DATA_PKL = None
MODEL_DIR = None
MODEL_SRC = None            # path of legal_gqa_model.py
RESUME_DIR = None           # output folder of an earlier run of THIS notebook
OUT_DIR = None              # None = /kaggle/working/<run name>_submission
# [[END OF SETTINGS]]

for _name in ["DATA_PKL", "MODEL_DIR", "MODEL_SRC", "RESUME_DIR", "OUT_DIR"]:
    _value = os.environ.get("RR_" + _name)
    if _value:
        globals()[_name] = _value
if os.environ.get("RR_SMOKE") == "1":
    SMOKE_TEST = True
# (more env overrides, only used for local testing)
for _name in ["PROTOCOL", "HEAD", "CLASS_WEIGHTS", "ADAPTER", "RUN_NAME", "LORMA_VARIANT"]:
    _value = os.environ.get("RR_" + _name)
    if _value:
        globals()[_name] = _value
if os.environ.get("RR_MAX_HOURS"):
    MAX_SESSION_HOURS = float(os.environ["RR_MAX_HOURS"])
if os.environ.get("RR_FEATURES") == "0":
    USE_FEATURES = False
if os.environ.get("RR_EPOCHS"):
    EPOCHS = int(os.environ["RR_EPOCHS"])

assert ADAPTER in ("lora", "lorma"), "ADAPTER must be lora or lorma"
if RUN_NAME is None:
    RUN_NAME = "rr_model1_gqa_lora" if ADAPTER == "lora" else "rr_model1_gqa_lorma"
EXPERIMENT_NAME = RUN_NAME
if OUT_DIR is None:
    OUT_DIR = "/kaggle/working/" + RUN_NAME + "_submission"

MAX_DOCS = None
if SMOKE_TEST:
    EPOCHS = 2
    PATIENCE = 2
    N_FOLDS = 2
    BATCH_CHUNKS = 2
    MICRO_BATCH = 1
    EVAL_BATCH = 2
    TRAIN_CHUNK_MIN = 300
    TRAIN_CHUNK_MAX = 400
    INFER_CHUNK_LENS = [400, 300]
    HEAD_DIM = 64
    GRU_HIDDEN = 32
    LORA_R = 4
    LORMA_R = 4
    MAX_DOCS = 8
    TAU_GRID = [0.0, 0.5]
    ALPHA_GRID = [0.0, 0.5]

if os.environ.get("RR_EPOCHS"):
    EPOCHS = int(os.environ["RR_EPOCHS"])
    PATIENCE = max(PATIENCE, EPOCHS)

assert PROTOCOL in ("kfold", "dev")
assert HEAD in ("bigru", "linear")
assert CLASS_WEIGHTS in ("none", "sqrt", "inv")
os.makedirs(OUT_DIR, exist_ok=True)
print("SMOKE_TEST:", SMOKE_TEST, "| ADAPTER:", ADAPTER, "| run:", RUN_NAME, "| PROTOCOL:", PROTOCOL, "| OUT_DIR:", OUT_DIR)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

if ADAPTER == "lora":
    try:
        import peft
    except ImportError:
        os.system("pip install -q peft")
        import peft
    from peft import LoraConfig, get_peft_model
    print("peft", peft.__version__, "| torch", torch.__version__)

    # Kaggle images can ship a torchao that is too old for the installed peft, which makes get_peft_model crash.
    # We never use torchao, so tell peft it is not available.
    try:
        peft.import_utils.is_torchao_available()
    except ImportError as err:
        print("peft/torchao version clash ignored:", str(err)[:100])
        def _torchao_off():
            return False
        for _name, _mod in list(sys.modules.items()):
            if _name.startswith("peft") and hasattr(_mod, "is_torchao_available"):
                _mod.is_torchao_available = _torchao_off

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("device:", DEVICE)
if DEVICE.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def time_is_up():
    return (time.time() - T0) > MAX_SESSION_HOURS * 3600

## 2. Find data, model code and Phase-1 weights

In [ ]:
def find_files(filename):
    return sorted(glob.glob("/kaggle/input/**/" + filename, recursive=True))


# ---------- data ----------
if DATA_PKL is None:
    candidates = find_files("tokenized_rr.pkl")
    if len(candidates) == 0:
        raise FileNotFoundError("tokenized_rr.pkl not found. Attach the output of the RR data-prep notebook.")
    if len(candidates) == 1:
        DATA_PKL = candidates[0]
    else:
        # several data-prep outputs attached: take the one with the most test sentences, then the newest
        print("More than one tokenized_rr.pkl found:")
        ranked = []
        for p in candidates:
            fh = open(p, "rb")
            b = pickle.load(fh)
            fh.close()
            n_test = 0
            for dom in b["meta"]["domains"]:
                for doc in b["data"][dom]["test"]["docs"]:
                    n_test += len(doc["sentences"])
            print("   ", p, "| test sentences:", n_test)
            ranked.append((n_test, os.path.getmtime(p), p))
            del b
        ranked.sort()
        DATA_PKL = ranked[-1][2]
print("data:", DATA_PKL)

DATA_DIR = os.path.dirname(DATA_PKL)
utils_candidates = [os.path.join(DATA_DIR, "rr_utils.py")] + find_files("rr_utils.py")
UTILS_PATH = None
for p in utils_candidates:
    if os.path.exists(p):
        UTILS_PATH = p
        break
if UTILS_PATH is None:
    raise FileNotFoundError("rr_utils.py not found next to tokenized_rr.pkl.")
sys.path.insert(0, os.path.dirname(UTILS_PATH))
import rr_utils
print("rr_utils:", UTILS_PATH)


# ---------- Phase-1 weights ----------
def find_model_dir():
    if MODEL_DIR is not None:
        return MODEL_DIR
    found = []
    for cfg_path in find_files("config.json"):
        folder = os.path.dirname(cfg_path)
        try:
            model_type = json.load(open(cfg_path)).get("model_type")
        except Exception:
            continue
        has_weights = os.path.exists(os.path.join(folder, "model.safetensors")) or \
                      os.path.exists(os.path.join(folder, "pytorch_model.bin"))
        if model_type == "legal_gqa" and has_weights:
            found.append(folder)
    if len(found) == 0:
        raise FileNotFoundError("No Phase-1 Model-1 folder (model_type legal_gqa). Attach the Model-1 notebook output.")
    final_ones = []
    for d in found:
        if "checkpoint-" not in d:
            final_ones.append(d)
    if len(final_ones) > 0:
        return final_ones[0]
    return max(found, key=lambda d: int(d.rsplit("-", 1)[1]))


MODEL_PATH = find_model_dir()
print("Phase-1 model:", MODEL_PATH)

# ---------- model code ----------
def find_model_src():
    if MODEL_SRC is not None:
        return MODEL_SRC
    places = ["/kaggle/working/legal_gqa_model.py", os.path.join(MODEL_PATH, "legal_gqa_model.py"),
              os.path.join(os.path.dirname(MODEL_PATH), "legal_gqa_model.py")]
    places = places + find_files("legal_gqa_model.py")
    for p in places:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(
        "legal_gqa_model.py not found. Attach an input that contains it (for example the output of your "
        "Phase-2 Model-1 NER notebook, which wrote it), or put it in a small dataset.")


SRC_PATH = find_model_src()
print("model code:", SRC_PATH)
sys.path.insert(0, os.path.dirname(SRC_PATH))
from legal_gqa_model import LegalGQAConfig, LegalGQAForCausalLM

# ---------- resume from an earlier run of this notebook ----------
def copy_missing(src, dst):
    for root, dirs, files in os.walk(src):
        rel = os.path.relpath(root, src)
        target = os.path.join(dst, rel)
        os.makedirs(target, exist_ok=True)
        for name in files:
            out_path = os.path.join(target, name)
            if not os.path.exists(out_path):
                shutil.copy(os.path.join(root, name), out_path)


resume_src = RESUME_DIR
if resume_src is None:
    prev = find_files(EXPERIMENT_NAME + "/cv_state.json")
    if len(prev) > 0:
        resume_src = os.path.dirname(prev[-1])
if resume_src is not None and os.path.abspath(resume_src) != os.path.abspath(OUT_DIR):
    copy_missing(resume_src, OUT_DIR)
    print("Trained run copied from:", resume_src)
else:
    raise FileNotFoundError("Training run '" + EXPERIMENT_NAME + "' (fold folders + cv_state.json) not found in /kaggle/input. "
                            "Attach the output of the training notebook, or check ADAPTER / RUN_NAME.")

## 3. Load documents and build features

In [ ]:
f = open(DATA_PKL, "rb")
blob = pickle.load(f)
f.close()
meta = blob["meta"]
data = blob["data"]

label2id = meta["label2id"]
K = int(meta["num_labels"])
id2label = {}
for name in label2id:
    id2label[label2id[name]] = name
LABEL_NAMES = []
for c in range(K):
    LABEL_NAMES.append(id2label[c])

DOMAINS = meta["domains"]
DOMAIN_IDX = {}
for i in range(len(DOMAINS)):
    DOMAIN_IDX[DOMAINS[i]] = i
SEP_ID = int(meta["sep_id"])
PAD_ID = int(meta["pad_id"])
HEADER_IDS = meta["header_ids"]
MAX_SENT_TOK = int(meta["max_sent_tok"])
DOC_CHUNK_CTX = int(meta["doc_chunk_ctx"])
print("labels:", K, "| domains:", DOMAINS, "| SEP:", SEP_ID, "| PAD:", PAD_ID)

# ---------- legal cue features ----------
CUES = [
    ("versus", r"\s(?:v|vs)\.?\s+[A-Z(]", 0),
    ("reporter", r"\b(?:AIR|SCC|SCR|ITR|ITD|CLJ|Taxman|SCALE|WLR|All ER|Comp\.? ?Cas)\b", 0),
    ("year_bracket", r"[\(\[]\s*(?:18|19|20)\d\d\s*[\)\]]", 0),
    ("section", r"\bsections?\b|\bsec\.|\bs\.\s?\d", re.I),
    ("article", r"\barticles?\s+\d", re.I),
    ("act_year", r"\bAct\b,?\s+(?:of\s+)?(?:18|19|20)\d\d", 0),
    ("rule_reg", r"\b(?:rules?|regulations?|clauses?|sub-?sections?|proviso|schedule)\b", re.I),
    ("held", r"\bheld\b", re.I),
    ("conclude", r"\b(?:we hold|we are of the (?:view|opinion)|in our (?:view|opinion)|we find|we conclude|accordingly|therefore|thus|hence)\b", re.I),
    ("disposal", r"\b(?:dismiss|allowed|set aside|quash|disposed|remand|affirm|upheld)\w*", re.I),
    ("argue", r"\b(?:contend|submit|argu|urge|learned (?:senior )?counsel|learned advocate)\w*", re.I),
    ("petitioner_side", r"\b(?:petitioner|appellant|plaintiff|applicant|assessee|complainant)s?\b", re.I),
    ("respondent_side", r"\b(?:respondent|defendant|revenue|department|opposite part(?:y|ies)|accused)s?\b", re.I),
    ("issue_q", r"\b(?:whether|question|issue|points? for|framed)\b", re.I),
    ("dissent", r"\b(?:dissent|minority view|respectfully|I disagree|unable to agree)\b", re.I),
    ("overrule", r"\b(?:overrul|reversed|no longer good law|wrongly decided|bad in law)\w*", re.I),
    ("distinguish", r"\b(?:distinguish|not applicable|of no assistance|not attracted|inapplicable)\w*", re.I),
    ("relied", r"\b(?:relied|reliance|followed|approved|reiterated|affirmed|referred)\b", re.I),
    ("lower_court", r"\b(?:high court|tribunal|trial court|lower court|learned (?:judge|single|magistrate)|impugned|appellate|commissioner)\b", re.I),
    ("facts_verbs", r"\b(?:filed|entered into|agreed|incorporated|appointed|issued|passed|notice|dated)\b", re.I),
    ("list_marker", r"^\(?(?:\d+|[ivxlcIVXLC]+|[a-zA-Z])[\.\)]\s", 0),
    ("has_quote", r"[\"“”]", 0),
    ("money", r"(?:Rs\.?|₹|INR|\$)\s?[\d,]+|\blakhs?\b|\bcrores?\b", re.I),
    ("date", r"\b\d{1,2}[./-]\d{1,2}[./-]\d{2,4}\b|\b(?:January|February|March|April|May|June|July|August|September|October|November|December)\b", 0),
]
CUE_REGEX = []
for _name, _pattern, _flags in CUES:
    CUE_REGEX.append(re.compile(_pattern, _flags))
N_CUE = len(CUES) + 5          # regex cues + 5 numeric cues
N_POS = 4
FEAT_DIM = N_CUE + N_POS


def sentence_cue_features(text, n_tokens):
    vec = np.zeros(N_CUE, dtype=np.float32)
    k = 0
    for rx in CUE_REGEX:
        if rx.search(text) is not None:
            vec[k] = 1.0
        k += 1
    n_chars = max(len(text), 1)
    digits = 0
    uppers = 0
    for ch in text:
        if ch.isdigit():
            digits += 1
        if ch.isupper():
            uppers += 1
    vec[k] = digits / n_chars
    vec[k + 1] = uppers / n_chars
    vec[k + 2] = math.log1p(n_tokens) / 6.0
    if text.endswith("?") or text.endswith(":"):
        vec[k + 3] = 1.0
    pet = len(re.findall(r"\b(?:petitioner|appellant|plaintiff|applicant|assessee)s?\b", text, re.I))
    res = len(re.findall(r"\b(?:respondent|defendant|revenue|department)s?\b", text, re.I))
    if pet > res:
        vec[k + 4] = 1.0
    elif res > pet:
        vec[k + 4] = -1.0
    return vec


def build_doc_features(sentences, sent_ids):
    n = len(sentences)
    feats = np.zeros((n, FEAT_DIM), dtype=np.float32)
    norm = math.log1p(2000)
    for i in range(n):
        feats[i, :N_CUE] = sentence_cue_features(sentences[i], len(sent_ids[i]))
        feats[i, N_CUE + 0] = i / max(n - 1, 1)
        feats[i, N_CUE + 1] = math.log1p(i) / norm
        feats[i, N_CUE + 2] = math.log1p(n - 1 - i) / norm
        feats[i, N_CUE + 3] = math.log1p(n) / norm
    return feats


# ---------- document table ----------
DOCS = []          # labelled documents first, then test documents
for domain in DOMAINS:
    for split in ["train", "dev"]:
        for doc in data[domain][split]["docs"]:
            DOCS.append({"domain": domain, "split": split, "id": doc["id"], "sentences": doc["sentences"],
                         "sent_ids": doc["sent_ids"], "labels": doc["labels"], "n": len(doc["sentences"])})
if MAX_DOCS is not None:
    # smoke test: keep only a few documents of each (domain, split)
    keep = []
    seen = {}
    for d in DOCS:
        key = d["domain"] + d["split"]
        seen[key] = seen.get(key, 0) + 1
        if d["split"] == "train" and seen[key] <= MAX_DOCS - 3:
            keep.append(d)
        elif d["split"] == "dev" and seen[key] <= 3:
            keep.append(d)
    DOCS = keep
N_LAB = len(DOCS)

for domain in DOMAINS:
    for doc in data[domain]["test"]["docs"]:
        DOCS.append({"domain": domain, "split": "test", "id": doc["id"], "sentences": doc["sentences"],
                     "sent_ids": doc["sent_ids"], "labels": None, "n": len(doc["sentences"])})
N_TEST = len(DOCS) - N_LAB

t1 = time.time()
for d in DOCS:
    d["feats"] = build_doc_features(d["sentences"], d["sent_ids"])
print("labelled docs:", N_LAB, "| test docs:", N_TEST, "| feature dim:", FEAT_DIM, "| %.1fs" % (time.time() - t1))
if N_TEST == 0:
    print("NOTE: no test documents in this data file (test set not released yet). Training and OOF still run.")

TEST_IDX = list(range(N_LAB, len(DOCS)))

## 4. Metrics, class weights, prior correction and transition-Viterbi

In [ ]:
def confusion_matrix(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    flat = np.bincount(y_true * K + y_pred, minlength=K * K)
    return flat.reshape(K, K)


def scores_from_conf(conf):
    tp = np.diag(conf).astype(np.float64)
    fp = conf.sum(axis=0) - tp
    fn = conf.sum(axis=1) - tp
    precision = np.zeros(K)
    recall = np.zeros(K)
    f1 = np.zeros(K)
    for c in range(K):
        if tp[c] + fp[c] > 0:
            precision[c] = tp[c] / (tp[c] + fp[c])
        if tp[c] + fn[c] > 0:
            recall[c] = tp[c] / (tp[c] + fn[c])
        if precision[c] + recall[c] > 0:
            f1[c] = 2 * precision[c] * recall[c] / (precision[c] + recall[c])
    support = conf.sum(axis=1)
    return precision, recall, f1, support


def macro_from_conf(conf):
    precision, recall, f1, support = scores_from_conf(conf)
    present = support > 0
    out = {}
    out["f1_present"] = float(f1[present].mean()) if present.any() else 0.0
    out["f1_all13"] = float(f1.mean())
    out["precision_present"] = float(precision[present].mean()) if present.any() else 0.0
    out["recall_present"] = float(recall[present].mean()) if present.any() else 0.0
    out["precision_all13"] = float(precision.mean())
    out["recall_all13"] = float(recall.mean())
    return out


def class_counts(doc_idx_list):
    counts = np.zeros(K, dtype=np.float64)
    for di in doc_idx_list:
        counts += np.bincount(DOCS[di]["labels"], minlength=K)
    return counts


def make_class_weights(counts):
    w = np.zeros(K, dtype=np.float64)
    total = counts.sum()
    k_seen = float((counts > 0).sum())
    if CLASS_WEIGHTS == "none":
        for c in range(K):
            if counts[c] > 0:
                w[c] = 1.0
        return w
    for c in range(K):
        if counts[c] > 0:
            base = total / (k_seen * counts[c])
            if CLASS_WEIGHTS == "sqrt":
                w[c] = math.sqrt(base)
            else:
                w[c] = base
    # rescale so the average weight over training sentences is 1, then cap
    scale = total / (w * counts).sum()
    w = w * scale
    for c in range(K):
        if w[c] > WEIGHT_CAP:
            w[c] = WEIGHT_CAP
    return w


def log_prior_from_counts(counts):
    p = (counts + 1.0) / (counts.sum() + K)
    return np.log(p)


def log_transitions(doc_idx_list, domain):
    trans = np.zeros((K, K), dtype=np.float64)
    for di in doc_idx_list:
        d = DOCS[di]
        if d["domain"] != domain:
            continue
        lab = d["labels"]
        for i in range(len(lab) - 1):
            trans[lab[i], lab[i + 1]] += 1
    trans = trans + 0.5
    trans = trans / trans.sum(axis=1, keepdims=True)
    return np.log(trans)


def decode(logp, trans_log, log_prior, tau, alpha):
    emission = logp - tau * log_prior[None, :]
    n = emission.shape[0]
    if alpha == 0.0 or n == 1:
        return emission.argmax(axis=1)
    score = emission[0].copy()
    back = np.zeros((n, K), dtype=np.int64)
    pair = alpha * trans_log
    for i in range(1, n):
        cand = score[:, None] + pair
        back[i] = cand.argmax(axis=0)
        score = cand.max(axis=0) + emission[i]
    path = np.zeros(n, dtype=np.int64)
    path[n - 1] = int(score.argmax())
    for i in range(n - 1, 0, -1):
        path[i - 1] = back[i, path[i]]
    return path


# self-test of the helpers
_c = confusion_matrix([0, 1, 1, 2], [0, 1, 2, 2])
assert macro_from_conf(_c)["f1_present"] > 0
_lp = np.log(np.full((5, K), 1.0 / K))
assert len(decode(_lp, np.log(np.full((K, K), 1.0 / K)), np.log(np.full(K, 1.0 / K)), 0.0, 0.5)) == 5
print("helpers OK")

## 5. Chunks and batching

In [ ]:
def make_train_chunks(doc_idx_list, rng, rare_labels):
    chunks = []
    for di in doc_idx_list:
        d = DOCS[di]
        max_len = rng.randint(TRAIN_CHUNK_MIN, TRAIN_CHUNK_MAX)
        parts = rr_utils.build_doc_chunks(d["sent_ids"], d["labels"], HEADER_IDS[d["domain"]], SEP_ID,
                                          max_len, DOC_CHUNK_CTX, MAX_SENT_TOK)
        for c in parts:
            c["doc"] = di
            chunks.append(c)

    extra = []
    if OVERSAMPLE_RARE and len(rare_labels) > 0:
        for c in chunks:
            if np.isin(c["labels"], rare_labels).any():
                for _ in range(OVERSAMPLE_EXTRA):
                    extra.append(c)
    chunks = chunks + extra
    rng.shuffle(chunks)
    return chunks


def collate(chunks):
    B = len(chunks)
    T = 0
    S = 0
    for c in chunks:
        T = max(T, len(c["input_ids"]))
        S = max(S, len(c["end_pos"]))

    ids = np.full((B, T), PAD_ID, dtype=np.int64)
    end = np.zeros((B, S), dtype=np.int64)
    start = np.zeros((B, S), dtype=np.int64)
    smask = np.zeros((B, S), dtype=bool)
    labels = np.full((B, S), -100, dtype=np.int64)
    feats = np.zeros((B, S, FEAT_DIM), dtype=np.float32)
    dom = np.zeros(B, dtype=np.int64)

    for b in range(B):
        c = chunks[b]
        d = DOCS[c["doc"]]
        header_len = 1 + len(HEADER_IDS[d["domain"]])
        ids[b, :len(c["input_ids"])] = c["input_ids"]
        n_s = len(c["end_pos"])
        end[b, :n_s] = c["end_pos"]
        for k in range(n_s):
            if k == 0:
                start[b, k] = header_len
            else:
                start[b, k] = c["end_pos"][k - 1] + 1
        smask[b, :n_s] = True
        labels[b, :n_s] = c["labels"]
        feats[b, :n_s, :] = d["feats"][c["sent_idx"]]
        dom[b] = DOMAIN_IDX[d["domain"]]

    batch = {}
    batch["input_ids"] = torch.from_numpy(ids)
    batch["end"] = torch.from_numpy(end)
    batch["start"] = torch.from_numpy(start)
    batch["smask"] = torch.from_numpy(smask)
    batch["labels"] = torch.from_numpy(labels)
    batch["feats"] = torch.from_numpy(feats)
    batch["dom"] = torch.from_numpy(dom)
    return batch


def to_device(batch):
    out = {}
    for key in batch:
        out[key] = batch[key].to(DEVICE)
    return out


# quick self-test on the first labelled document
_d0 = DOCS[0]
_parts = rr_utils.build_doc_chunks(_d0["sent_ids"], _d0["labels"], HEADER_IDS[_d0["domain"]], SEP_ID,
                                   INFER_CHUNK_LENS[-1], DOC_CHUNK_CTX, MAX_SENT_TOK)
for _c in _parts:
    _c["doc"] = 0
_b = collate(_parts[:2])
for _row in range(_b["input_ids"].shape[0]):
    for _k in range(int(_b["smask"][_row].sum())):
        assert int(_b["input_ids"][_row, int(_b["end"][_row, _k])]) == SEP_ID
        assert int(_b["start"][_row, _k]) <= int(_b["end"][_row, _k])
print("collate OK | chunks in first doc:", len(_parts), "| batch shape", tuple(_b["input_ids"].shape))

## 6. Model: Phase-1 backbone + adapter (LoRA or LoRMA) + sentence head

In [ ]:
from torch.utils.checkpoint import checkpoint as torch_checkpoint


# ---------------------------------------------------------------- LoRMA layer (written by hand, PEFT has none)
class LoRMALinear(nn.Module):
    """Frozen linear layer W0 (d_out x d_in) with a low-rank MULTIPLICATIVE update on its output side:
         plus:  W = ((alpha/r) B A + I) W0     B: d_out x r (zeros at start), A: r x d_out (random)
         perm:  W = P(B A) W0                  P rotates row i of B A cyclically by i; starts at P(BA) = I
       so training starts exactly at the Phase-1 weights."""
    def __init__(self, base, r, alpha, dropout, variant):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        d = base.out_features
        self.d = d
        self.r = r
        self.variant = variant
        self.scale = alpha / r
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.lorma_A = nn.Parameter(torch.zeros(r, d))
        self.lorma_B = nn.Parameter(torch.zeros(d, r))
        if variant == "plus":
            nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))      # B stays 0 -> (scale*BA + I) = I
        else:
            with torch.no_grad():
                self.lorma_B.normal_(0.0, 0.02)
                self.lorma_B[:, 0] = 1.0                                 # first column of B = ones, rest random
                self.lorma_A.zero_()
                self.lorma_A[0, 0] = 1.0                                 # A[0,0] = 1, rest 0 -> BA has ones in column 0 only
            ii = torch.arange(d).unsqueeze(1)
            jj = torch.arange(d).unsqueeze(0)
            self.register_buffer("rot_idx", (jj - ii) % d, persistent=False)    # rotated[i, j] = M[i, (j - i) mod d]

    def forward(self, x):
        if self.variant == "plus":
            z = self.base(x)                                             # W0 x
            return z + self.scale * F.linear(F.linear(self.drop(z), self.lorma_A), self.lorma_B)
        M = self.lorma_B @ self.lorma_A                                  # d x d, rank <= r
        R = M.gather(1, self.rot_idx)                                    # rank-inflated by the row rotation
        w = R.to(self.base.weight.dtype) @ self.base.weight              # P(BA) W0
        b = None if self.base.bias is None else R.to(self.base.bias.dtype) @ self.base.bias
        return F.linear(self.drop(x), w, b)


def add_lorma(root, targets, r, alpha, dropout, variant):
    names = []
    for name, module in list(root.named_modules()):
        for child_name, child in list(module.named_children()):
            if child_name in targets and isinstance(child, nn.Linear):
                setattr(module, child_name, LoRMALinear(child, r, alpha, dropout, variant))
                names.append(name + "." + child_name)
    return names



class RRModel(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        d = backbone.config.n_embd
        self.d = d

        self.sent_norm = nn.LayerNorm(2 * d)
        self.sent_drop = nn.Dropout(HEAD_DROPOUT)
        self.sent_proj = nn.Linear(2 * d, HEAD_DIM)

        self.feat_proj = nn.Linear(FEAT_DIM, 64)
        self.dom_emb = nn.Embedding(len(DOMAINS), DOMAIN_EMB)
        self.mix = nn.Linear(HEAD_DIM + 64 + DOMAIN_EMB, HEAD_DIM)
        self.mix_drop = nn.Dropout(HEAD_DROPOUT)

        if HEAD == "bigru":
            self.gru = nn.GRU(HEAD_DIM, GRU_HIDDEN, batch_first=True, bidirectional=True)
            out_dim = 2 * GRU_HIDDEN + HEAD_DIM
        else:
            out_dim = HEAD_DIM
        self.out_drop = nn.Dropout(HEAD_DROPOUT)
        self.cls = nn.Linear(out_dim, K)

    def hidden(self, input_ids):
        # same computation as the Phase-1 forward(), stopping before lm_head
        b = self.backbone
        T = input_ids.shape[1]
        x = b.drop(b.wte(input_ids))
        cos, sin = b._rope_slice(0, T, input_ids.device, x.dtype)
        for blk in b.blocks:
            if self.training and USE_CHECKPOINT:
                def run(h, blk=blk):
                    return blk(h, cos, sin, None, True, None, False)[0]
                x = torch_checkpoint(run, x, use_reentrant=False)
            else:
                x = blk(x, cos, sin, None, True, None, False)[0]     # causal; right padding needs no mask
        return b.ln_f(x)

    def forward(self, batch):
        ids = batch["input_ids"]
        end = batch["end"]
        start = batch["start"]
        smask = batch["smask"]

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            h = self.hidden(ids)
        h = h.float()
        B, T, d = h.shape
        S = end.shape[1]

        # SEP state of every sentence
        sep_state = torch.gather(h, 1, end.unsqueeze(-1).expand(B, S, d))

        # mean of the sentence's own tokens: tokens start .. end-1  (cumulative-sum trick)
        zeros = torch.zeros(B, 1, d, device=h.device, dtype=h.dtype)
        cs = torch.cat([zeros, torch.cumsum(h, dim=1)], dim=1)
        cs_end = torch.gather(cs, 1, end.unsqueeze(-1).expand(B, S, d))
        cs_start = torch.gather(cs, 1, start.unsqueeze(-1).expand(B, S, d))
        length = (end - start).clamp(min=1).unsqueeze(-1).float()
        mean_state = (cs_end - cs_start) / length

        x = torch.cat([sep_state, mean_state], dim=-1)
        x = self.sent_drop(self.sent_norm(x))
        x = F.gelu(self.sent_proj(x))

        if USE_FEATURES:
            f = F.gelu(self.feat_proj(batch["feats"]))
        else:
            f = torch.zeros(B, S, 64, device=h.device, dtype=h.dtype)
        dm = self.dom_emb(batch["dom"]).unsqueeze(1).expand(B, S, DOMAIN_EMB)
        x = F.gelu(self.mix(torch.cat([x, f, dm], dim=-1)))
        x = self.mix_drop(x)

        if HEAD == "bigru":
            lengths = smask.sum(dim=1).cpu()
            packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
            out, _ = self.gru(packed)
            out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=S)
            out = torch.cat([out, x], dim=-1)
        else:
            out = x
        return self.cls(self.out_drop(out)).float()


LOAD_REPORTED = False


def build_model(identity_check=False):
    global LOAD_REPORTED
    backbone, info = LegalGQAForCausalLM.from_pretrained(MODEL_PATH, output_loading_info=True)
    if not LOAD_REPORTED:
        missing = []
        for key in info.get("missing_keys", []):
            if key != "lm_head.weight":          # tied to wte
                missing.append(key)
        unexpected = list(info.get("unexpected_keys", []))
        print("Phase-1 weights loaded | missing keys:", len(missing), "| unexpected keys:", len(unexpected))
        if len(missing) > 0:
            raise RuntimeError("Phase-1 weights were NOT fully loaded, missing: " + str(missing[:5]))
        if backbone.config.vocab_size <= max(SEP_ID, 0):
            raise RuntimeError("tokenizer / model vocab mismatch")
        LOAD_REPORTED = True
    backbone.config.use_cache = False

    if ADAPTER == "lorma":
        backbone.float()
    if identity_check:
        probe = RRModel.__new__(RRModel)
        nn.Module.__init__(probe)
        probe.backbone = backbone
        ids = torch.randint(0, backbone.config.vocab_size, (1, 40))
        backbone.eval()
        with torch.no_grad():
            h_before = RRModel.hidden(probe, ids).clone()

    if ADAPTER == "lora":
        # some peft / transformers versions ask the model for its input embedding, give it a safe answer
        try:
            backbone.get_input_embeddings()
        except NotImplementedError:
            backbone.get_input_embeddings = types.MethodType(lambda self: self.wte, backbone)

        lora_cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                              target_modules=LORA_TARGETS, bias="none")
        get_peft_model(backbone, lora_cfg)          # wraps the linear layers of `backbone` in place and freezes the base
        n_wrapped = 0
        for name, _ in backbone.named_modules():
            if name.endswith("lora_A"):
                n_wrapped += 1
        if n_wrapped == 0:
            raise RuntimeError("LoRA wrapped no layers, check LORA_TARGETS")

        if identity_check:
            with torch.no_grad():
                h_after = RRModel.hidden(probe, ids)
            diff = float((h_before - h_after).abs().max())
            print("LoRA wrapped", n_wrapped, "linear layers | identity check max |diff| vs Phase-1:", "%.2e" % diff)
            if diff > 1e-3:
                raise RuntimeError("LoRA layers do not start at the identity")
    else:
        for p in backbone.parameters():
            p.requires_grad = False                 # the Phase-1 backbone stays frozen
        wrapped = add_lorma(backbone, set(LORMA_TARGETS), LORMA_R, LORMA_ALPHA, LORMA_DROPOUT, LORMA_VARIANT)
        if len(wrapped) == 0:
            raise RuntimeError("LoRMA wrapped no layers, check LORMA_TARGETS")
        if identity_check:
            backbone.eval()
            with torch.no_grad():
                h_after = RRModel.hidden(probe, ids)
            diff = float((h_before - h_after).abs().max())
            print("LoRMA (%s) wrapped %d linear layers | identity check max |diff| vs Phase-1: %.2e" % (LORMA_VARIANT, len(wrapped), diff))
            if diff > 1e-3:
                raise RuntimeError("LoRMA layers do not start at the identity (diff %.3e)" % diff)

    model = RRModel(backbone)
    model.to(DEVICE)
    return model


def trainable_names(model):
    names = []
    for name, p in model.named_parameters():
        if p.requires_grad:
            names.append(name)
    return names


def get_trainable_state(model):
    state = {}
    for name, p in model.named_parameters():
        if p.requires_grad:
            state[name] = p.detach().cpu().clone()
    return state


def load_trainable_state(model, state):
    for name, p in model.named_parameters():
        if name in state:
            p.data.copy_(state[name].to(p.device))


_m = build_model(identity_check=True)
total = 0
trainable = 0
for _n, _p in _m.named_parameters():
    total += _p.numel()
    if _p.requires_grad:
        trainable += _p.numel()
print("total %.2fM | trainable %.2fM (%.2f%%)" % (total / 1e6, trainable / 1e6, 100.0 * trainable / total))
del _m
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

## 7. Prediction

In [ ]:
def predict_docs(model, doc_list, chunk_lens):
    """Average log-probabilities of several different chunk cuttings. Returns {doc_index: (n, K) array}."""
    model.eval()
    jobs = []
    acc = {}
    cnt = {}
    for di in doc_list:
        d = DOCS[di]
        acc[di] = np.zeros((d["n"], K), dtype=np.float64)
        cnt[di] = np.zeros(d["n"], dtype=np.float64)
        for L in chunk_lens:
            parts = rr_utils.build_doc_chunks(d["sent_ids"], None, HEADER_IDS[d["domain"]], SEP_ID,
                                              L, DOC_CHUNK_CTX, MAX_SENT_TOK)
            for c in parts:
                c["doc"] = di
                jobs.append(c)
    jobs.sort(key=lambda c: len(c["input_ids"]))

    pos = 0
    while pos < len(jobs):
        batch_chunks = jobs[pos:pos + EVAL_BATCH]
        batch = to_device(collate(batch_chunks))
        with torch.no_grad():
            em = model(batch)
            logp = torch.log_softmax(em, dim=-1).cpu().numpy()
        for b in range(len(batch_chunks)):
            c = batch_chunks[b]
            for k in range(len(c["end_pos"])):
                if c["is_target"][k]:
                    s = int(c["sent_idx"][k])
                    acc[c["doc"]][s] += logp[b, k]
                    cnt[c["doc"]][s] += 1
        pos += EVAL_BATCH

    out = {}
    for di in doc_list:
        if (cnt[di] < 1).any():
            raise RuntimeError("some sentences got no prediction in doc " + DOCS[di]["id"])
        avg = acc[di] / cnt[di][:, None]
        m = avg.max(axis=1, keepdims=True)
        avg = avg - (m + np.log(np.exp(avg - m).sum(axis=1, keepdims=True)))
        out[di] = avg.astype(np.float32)
    return out

## 8. Load the fold models and predict the test documents

In [ ]:
# ---------- fold assignment ----------
def make_folds():
    labelled = list(range(N_LAB))
    if PROTOCOL == "dev":
        train_idx = []
        val_idx = []
        for i in labelled:
            if DOCS[i]["split"] == "dev":
                val_idx.append(i)
            else:
                train_idx.append(i)
        if len(val_idx) == 0:
            raise RuntimeError("no dev documents")
        return [(train_idx, val_idx)]

    rng = random.Random(SEED)
    fold_of = {}
    for domain in DOMAINS:
        idx = []
        for i in labelled:
            if DOCS[i]["domain"] == domain:
                idx.append(i)
        idx.sort(key=lambda i: -DOCS[i]["n"])          # big documents are spread evenly over the folds
        for block_start in range(0, len(idx), N_FOLDS):
            block = idx[block_start:block_start + N_FOLDS]
            order = list(range(N_FOLDS))
            rng.shuffle(order)
            for j in range(len(block)):
                fold_of[block[j]] = order[j]
    folds = []
    for k in range(N_FOLDS):
        train_idx = []
        val_idx = []
        for i in labelled:
            if fold_of[i] == k:
                val_idx.append(i)
            else:
                train_idx.append(i)
        if len(val_idx) == 0:
            raise RuntimeError("empty validation fold, lower N_FOLDS")
        folds.append((train_idx, val_idx))
    return folds


fingerprint_text = PROTOCOL + str(N_FOLDS) + str(SEED) + str(SMOKE_TEST)
for i in range(N_LAB):
    fingerprint_text += DOCS[i]["domain"] + DOCS[i]["id"] + str(DOCS[i]["n"])
FINGERPRINT = hashlib.md5(fingerprint_text.encode("utf-8")).hexdigest()

state_path = os.path.join(OUT_DIR, "cv_state.json")
if os.path.exists(state_path):
    old = json.load(open(state_path))
    if old.get("fingerprint") != FINGERPRINT:
        raise RuntimeError(
            "The labelled documents / protocol differ from the earlier run in OUT_DIR (the dataset may have changed). "
            "Remove the old fold folders or change EXPERIMENT_NAME to start a new run.")

FOLDS = make_folds()
with open(state_path, "w") as fh:
    json.dump({"fingerprint": FINGERPRINT, "protocol": PROTOCOL, "n_folds": len(FOLDS), "seed": SEED,
               "folds": [[DOCS[i]["id"] for i in v] for (t, v) in FOLDS]}, fh)
for k in range(len(FOLDS)):
    dom_counts = {}
    for i in FOLDS[k][1]:
        dom_counts[DOCS[i]["domain"]] = dom_counts.get(DOCS[i]["domain"], 0) + 1
    print("fold", k, "| train docs", len(FOLDS[k][0]), "| val docs", len(FOLDS[k][1]), dom_counts)


def save_pickle(obj, path):
    fh = open(path, "wb")
    pickle.dump(obj, fh, protocol=4)
    fh.close()


def load_pickle(path):
    fh = open(path, "rb")
    obj = pickle.load(fh)
    fh.close()
    return obj


ALL_DONE = True
for k in range(len(FOLDS)):
    train_idx, val_idx = FOLDS[k]
    fold_dir = os.path.join(OUT_DIR, "fold" + str(k))
    result_path = os.path.join(fold_dir, "fold_result.json")
    oof_path = os.path.join(fold_dir, "oof_logp.pkl")
    test_path = os.path.join(fold_dir, "test_logp.pkl")

    model = None
    if not os.path.exists(os.path.join(fold_dir, "best.pt")):
        raise FileNotFoundError("fold " + str(k) + ": best.pt not found in " + fold_dir + ". The training run is incomplete: "
                                "finish it with the training notebook first.")

    need_oof = not os.path.exists(oof_path)
    test_ids = [DOCS[i]["id"] for i in TEST_IDX]
    test_n = [DOCS[i]["n"] for i in TEST_IDX]
    need_test = False
    if N_TEST > 0:
        if not os.path.exists(test_path):
            need_test = True
        else:
            old_pack = load_pickle(test_path)
            if old_pack["ids"] != test_ids or ("n" in old_pack and old_pack["n"] != test_n):
                print("  fold", k, "old test predictions belong to a different test set, recomputing")
                need_test = True
    if need_oof or need_test:
        if model is None:
            set_seed(SEED + 100 * k)
            model = build_model(identity_check=False)
        best_state = torch.load(os.path.join(fold_dir, "best.pt"), map_location="cpu")
        load_trainable_state(model, best_state)
        if need_oof:
            t1 = time.time()
            oof = predict_docs(model, val_idx, INFER_CHUNK_LENS)
            save_pickle({"ids": [DOCS[i]["id"] for i in val_idx], "logp": oof}, oof_path)
            print("  fold", k, "OOF predictions saved (%.0fs)" % (time.time() - t1))
        if need_test:
            t1 = time.time()
            tst = predict_docs(model, TEST_IDX, INFER_CHUNK_LENS)
            save_pickle({"ids": test_ids, "n": test_n, "logp": tst}, test_path)
            print("  fold", k, "test predictions saved (%.0fs)" % (time.time() - t1))
    if model is not None:
        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

if ALL_DONE:
    print("\nAll fold models loaded.")
else:
    print("\nSTOPPED because of MAX_SESSION_HOURS. Progress is saved in", OUT_DIR)
    print("Attach this notebook's output as an input to a new run of the same notebook to continue.")

## 9. Out-of-fold results, tuning of prior correction and Viterbi

`tau` subtracts `tau * log(class prior)` from the log-probabilities (boosts rare labels when positive). `alpha` weights the label-transition log-probabilities in Viterbi decoding. Both are tuned for macro-F1 on the out-of-fold predictions, so they are not fitted on the training data of the model that produced them.

In [ ]:
if not ALL_DONE:
    print("Skipped: not all folds are finished yet.")
else:
    # ---------- collect OOF log-probs ----------
    oof = []     # entries: dict(doc, fold, logp)
    for k in range(len(FOLDS)):
        pack = load_pickle(os.path.join(OUT_DIR, "fold" + str(k), "oof_logp.pkl"))
        for di in FOLDS[k][1]:
            oof.append({"doc": di, "fold": k, "logp": pack["logp"][di]})

    # ---------- per-fold transitions and priors (from that fold's training docs only) ----------
    fold_trans = []
    fold_prior = []
    for k in range(len(FOLDS)):
        tr = {}
        for domain in DOMAINS:
            tr[domain] = log_transitions(FOLDS[k][0], domain)
        fold_trans.append(tr)
        fold_prior.append(log_prior_from_counts(class_counts(FOLDS[k][0])))

    def oof_confusion(tau, alpha, domain=None):
        conf = np.zeros((K, K), dtype=np.int64)
        for e in oof:
            d = DOCS[e["doc"]]
            if domain is not None and d["domain"] != domain:
                continue
            pred = decode(e["logp"], fold_trans[e["fold"]][d["domain"]], fold_prior[e["fold"]], tau, alpha)
            conf += confusion_matrix(d["labels"], pred)
        return conf

    base_conf = oof_confusion(0.0, 0.0)
    base = macro_from_conf(base_conf)
    print("OOF macro-F1 (argmax, no correction): present-labels %.4f | all-13 %.4f" % (base["f1_present"], base["f1_all13"]))

    results_grid = []
    best = None
    for tau in TAU_GRID:
        for alpha in ALPHA_GRID:
            sc = macro_from_conf(oof_confusion(tau, alpha))["f1_present"]
            results_grid.append((tau, alpha, sc))
            better = False
            if best is None or sc > best[2] + 1e-6:
                better = True
            elif abs(sc - best[2]) <= 1e-6 and (abs(tau) + alpha) < (abs(best[0]) + best[1]):
                better = True
            if better:
                best = (tau, alpha, sc)
    BEST_TAU, BEST_ALPHA, best_score = best
    print("grid (tau, alpha -> macro-F1):")
    for tau, alpha, sc in results_grid:
        print("   tau %5.2f  alpha %4.2f  ->  %.4f" % (tau, alpha, sc))
    print("BEST tau = %.2f, alpha = %.2f  ->  OOF macro-F1 (present) %.4f" % (BEST_TAU, BEST_ALPHA, best_score))

    final_conf = oof_confusion(BEST_TAU, BEST_ALPHA)
    final = macro_from_conf(final_conf)
    precision, recall, f1, support = scores_from_conf(final_conf)
    print("\nPooled OOF, after tuning: macro-F1 present %.4f | all-13 %.4f | macro-P %.4f | macro-R %.4f" % (
        final["f1_present"], final["f1_all13"], final["precision_present"], final["recall_present"]))
    print("\n%-24s %8s %8s %8s %8s" % ("label", "support", "prec", "recall", "F1"))
    for c in range(K):
        print("%-24s %8d %8.3f %8.3f %8.3f" % (LABEL_NAMES[c], support[c], precision[c], recall[c], f1[c]))

    per_domain = {}
    for domain in DOMAINS:
        conf_d = oof_confusion(BEST_TAU, BEST_ALPHA, domain)
        per_domain[domain] = macro_from_conf(conf_d)
        print("\n%s: macro-F1 present %.4f | all-13 %.4f | macro-P %.4f | macro-R %.4f" % (
            domain, per_domain[domain]["f1_present"], per_domain[domain]["f1_all13"],
            per_domain[domain]["precision_present"], per_domain[domain]["recall_present"]))
        p_d, r_d, f_d, s_d = scores_from_conf(conf_d)
        weak = []
        for c in range(K):
            if s_d[c] > 0:
                weak.append((f_d[c], LABEL_NAMES[c], int(s_d[c])))
        weak.sort()
        print("   weakest labels:", [(n, "%.2f" % v, s) for v, n, s in weak[:5]])

    # most frequent confusions
    pairs = []
    for a in range(K):
        for b in range(K):
            if a != b and final_conf[a, b] > 0:
                pairs.append((int(final_conf[a, b]), LABEL_NAMES[a], LABEL_NAMES[b]))
    pairs.sort(reverse=True)
    print("\nMost frequent confusions (true -> predicted):")
    for n, a, b in pairs[:10]:
        print("   %5d  %s -> %s" % (n, a, b))

    fold_summaries = []
    for k in range(len(FOLDS)):
        fold_summaries.append(json.load(open(os.path.join(OUT_DIR, "fold" + str(k), "fold_result.json"))))

    report = {
        "protocol": PROTOCOL, "n_folds": len(FOLDS),
        "oof_baseline_argmax": base, "best_tau": BEST_TAU, "best_alpha": BEST_ALPHA,
        "oof_tuned": final, "per_domain": per_domain,
        "per_class": {LABEL_NAMES[c]: {"support": int(support[c]), "precision": float(precision[c]),
                                       "recall": float(recall[c]), "f1": float(f1[c])} for c in range(K)},
        "top_confusions": [{"count": n, "true": a, "pred": b} for n, a, b in pairs[:25]],
        "confusion_matrix": final_conf.tolist(), "labels": LABEL_NAMES,
        "grid": [{"tau": t, "alpha": a, "macro_f1": s} for t, a, s in results_grid],
        "folds": fold_summaries,
        "settings": {"LORA_R": LORA_R, "LORA_ALPHA": LORA_ALPHA, "HEAD": HEAD, "USE_FEATURES": USE_FEATURES,
                     "CLASS_WEIGHTS": CLASS_WEIGHTS, "OVERSAMPLE_RARE": OVERSAMPLE_RARE, "EPOCHS": EPOCHS,
                     "LR_LORA": LR_LORA, "LR_HEAD": LR_HEAD, "INFER_CHUNK_LENS": INFER_CHUNK_LENS,
                     "SMOKE_TEST": SMOKE_TEST},
    }
    with open(os.path.join(OUT_DIR, "oof_report.json"), "w") as fh:
        json.dump(report, fh, indent=2)
    print("\nSaved oof_report.json")

## 10. Test predictions

In [ ]:
if not ALL_DONE:
    print("Skipped: not all folds are finished yet.")
elif N_TEST == 0:
    print("No test documents in the data file. Re-run the data-prep notebook after the test set is released,")
    print("attach its new output here and run this notebook again.")
else:
    # all labelled documents -> transitions and prior for the final decoding
    all_lab = list(range(N_LAB))
    final_trans = {}
    for domain in DOMAINS:
        final_trans[domain] = log_transitions(all_lab, domain)
    final_prior = log_prior_from_counts(class_counts(all_lab))

    packs = []
    for k in range(len(FOLDS)):
        packs.append(load_pickle(os.path.join(OUT_DIR, "fold" + str(k), "test_logp.pkl")))

    predictions = {}
    by_domain = {}
    for di in TEST_IDX:
        d = DOCS[di]
        total = np.zeros((d["n"], K), dtype=np.float64)
        for pack in packs:
            total += pack["logp"][di]
        avg = total / len(packs)
        m = avg.max(axis=1, keepdims=True)
        avg = avg - (m + np.log(np.exp(avg - m).sum(axis=1, keepdims=True)))
        path = decode(avg, final_trans[d["domain"]], final_prior, BEST_TAU, BEST_ALPHA)
        labels = []
        for c in path:
            labels.append(LABEL_NAMES[int(c)])
        if len(labels) != d["n"]:
            raise RuntimeError("length mismatch in " + d["id"])
        for lab in labels:
            if lab not in label2id:
                raise RuntimeError("label not in vocab: " + lab)
        predictions[d["id"]] = labels
        if d["domain"] not in by_domain:
            by_domain[d["domain"]] = {}
        by_domain[d["domain"]][d["id"]] = labels

    if len(predictions) != N_TEST:
        print("WARNING: document ids are not unique across CL and IT, the merged file has", len(predictions), "of", N_TEST)

    for domain in by_domain:
        with open(os.path.join(OUT_DIR, "prediction_" + domain + ".json"), "w") as fh:
            json.dump(by_domain[domain], fh)
    with open(os.path.join(OUT_DIR, "prediction.json"), "w") as fh:
        json.dump(predictions, fh)
    with zipfile.ZipFile(os.path.join(OUT_DIR, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(os.path.join(OUT_DIR, "prediction.json"), "prediction.json")

    counts = np.zeros(K, dtype=np.int64)
    for doc_id in predictions:
        for lab in predictions[doc_id]:
            counts[label2id[lab]] += 1
    print("test documents:", N_TEST, "| sentences:", int(counts.sum()))
    print("predicted label distribution:", {LABEL_NAMES[c]: int(counts[c]) for c in range(K) if counts[c] > 0})
    print("Saved prediction.json, prediction_<domain>.json and submission.zip in", OUT_DIR)
    print("Check the Codabench page for the exact file name / one-file-or-two-files rule of Phase 3.")

## Notes

- This notebook never trains. It needs the `best.pt` of every fold, so the training notebook must have finished all folds.
- The same notebook makes the submission for the LoRMA run (`ADAPTER = "lorma"`) and, later, for any model whose adapter code is added to section 6.
- Test predictions are cached per fold in `fold*/test_logp.pkl` together with the document ids, so an older (different) test set is detected and recomputed.
- Check the Codabench page for the exact file name / one-file-or-two-files rule of Phase 3 before uploading.